# Significance Tests — Task 1 F1 and Corrected Paired t-Tests

Recomputes the Task 1 column of Table I and every p-value the paper reports, straight from the
published inputs and results. Pull them from the Hub first (from the repository root):

```bash
mllmsent hub pull-datasets --groups inputs captions results
```

1. **Table I, Task 1** (GPT (OAI), DeepSeek): the published direct answers scored with the
   `kfold_task1_eval` protocol — `KFold(5, shuffle, seed 42)` over the images each model answered,
   weighted F1. The ± is the Student-t 95% CI over the five folds ($t_{0.975,4} \times \mathrm{SEM}$),
   the interval every table and figure of the paper reports.
2. **Significance**: the two-sided corrected resampled paired t-test (Nadeau & Bengio, 2003),
   $\operatorname{var}(\bar d) = \left(\tfrac{1}{k} + \tfrac{n_\text{test}}{n_\text{train}}\right) s_d^2$
   with $n_\text{test}/n_\text{train} = 1/(k-1)$ and $df = k-1$, Holm-adjusted within each family:
   - Task 1, GPT (OAI) vs DeepSeek, 4 settings (folds as in
     `src/mllmsent/evaluation/paired_ttest.py`: `StratifiedKFold(5, shuffle, 42)` over the images
     both models answered);
   - MLLMsent vs Swin and VADER (shared folds) and ResNet (Welch from the published mean and 95% CI,
     both variances inflated by the same factor), 10 comparisons (as in
     `src/mllmsent/evaluation/posthoc.py`).

   The uncorrected p-values are shown beside them for reference.

In [ ]:
import json
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import f1_score
from sklearn.model_selection import KFold, StratifiedKFold

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from mllmsent.evaluation.stats.hypothesis_tests import holm_correction

# ── Configuration ──────────────────────────────────────────────────────────────
DATA = ROOT / "data"
EXP = ROOT / "output" / "experiments"
RESULTS_JSON = ROOT / "output" / "reports" / "reports" / "results.json"

K = 5
RHO = 1 / (K - 1)                      # n_test / n_train in k-fold CV
INFLATION = (1 / K + RHO) / (1 / K)    # 2.25 for k = 5: t is divided by 1.5
SETTINGS = [("p5", 3), ("p3", 3), ("p5", 5), ("p3", 5)]
P5 = {"positive": 4, "slightlypositive": 3, "neutral": 2, "slightlynegative": 1, "negative": 0}
P3 = {"positive": 2, "neutral": 0, "negative": 1}
# GPT (OAI) direct answers per setting: (file, column)
OAI = {("p3", 3): ("alpha3p3.csv", "response"), ("p5", 3): ("alpha3p5.csv", "sentiment"),
       ("p3", 5): ("p3-results.csv", "response"), ("p5", 5): ("p5-results.csv", "response")}

## Helpers

In [ ]:
def label(value, sent_map):
    text = str(value).strip()
    if text.lstrip("-").isdigit():          # DeepSeek stores integer labels
        return int(text)
    return sent_map.get(text.replace(".", "").replace(" ", "").lower())


def answers(p, s):
    sent_map = P5 if p == "p5" else P3
    file, col = OAI[(p, s)]
    oai = pd.read_csv(DATA / "gpt4-openai-only" / file)
    ds = pd.read_csv(DATA / "deepseek-only" / f"alpha{s}{p}.csv")
    to_dict = lambda df, c: {i: lab for i, v in zip(df["id"], df[c]) if (lab := label(v, sent_map)) is not None}
    return to_dict(oai, col), to_dict(ds, "sentiment")


def fold_f1(gt, preds, splitter):
    ids, y = gt["id"].to_numpy(), gt["sentiment"].to_numpy()
    splits = splitter.split(gt, y) if isinstance(splitter, StratifiedKFold) else splitter.split(gt)
    return np.array([f1_score(y[te], [preds[i] for i in ids[te]], average="weighted") for _, te in splits])


def corrected_paired(a, b):
    d = np.asarray(a) - np.asarray(b)
    t_plain = d.mean() / (d.std(ddof=1) / math.sqrt(len(d)))
    t_nb = d.mean() / math.sqrt((1 / len(d) + RHO) * d.var(ddof=1))
    p = lambda t: 2 * stats.t.sf(abs(t), len(d) - 1)
    return d.mean(), p(t_plain), t_nb, p(t_nb)


def fold_scores(path):
    df = pd.read_csv(path)
    return (df.sort_values("kfold") if "kfold" in df.columns else df)["f1_score"].to_numpy(float)


def add_holm(df):
    df["p_holm_plain"] = holm_correction(df["p_plain"].tolist())[1]
    df["p_holm_corrected"] = holm_correction(df["p_corrected"].tolist())[1]
    return df

## 1. Table I, Task 1 — weighted F1 (mean ± 95% CI, %)

Each model is scored on the images it answered; the GPT (OAI) vs DeepSeek folds are drawn over the
images both answered.

In [ ]:
rows, task1 = [], []
for p, s in SETTINGS:
    gt = pd.read_csv(DATA / "gpt4-openai-classify" / f"percept_dataset_alpha{s}_{p}.csv")
    oai, ds = answers(p, s)
    for name, preds in (("GPT (OAI)", oai), ("DeepSeek", ds)):
        own = gt[gt["id"].isin(preds)].sort_values("id").reset_index(drop=True)
        f = fold_f1(own, preds, KFold(K, shuffle=True, random_state=42)) * 100
        rows.append(dict(setting=f"<s{s},{p.upper()}>", model=name, images=len(own),
                         f1=round(f.mean(), 1), pm_ci95=round(stats.t.ppf(0.975, K - 1) * stats.sem(f), 1)))
    common = gt[gt["id"].isin(set(oai) & set(ds))].sort_values("id").reset_index(drop=True)
    skf = StratifiedKFold(K, shuffle=True, random_state=42)
    diff, p_plain, t_nb, p_nb = corrected_paired(fold_f1(common, oai, skf), fold_f1(common, ds, skf))
    task1.append(dict(setting=f"<s{s},{p.upper()}>", images=len(common), diff=diff * 100,
                      t_corrected=t_nb, p_plain=p_plain, p_corrected=p_nb))

table1 = pd.DataFrame(rows)
table1

## 2. Task 1 — GPT (OAI) vs DeepSeek (Holm over 4)

In [ ]:
t1 = add_holm(pd.DataFrame(task1))
t1.round(4)

## 3. MLLMsent vs baselines (Holm over 10)

Swin and VADER share MLLMsent's folds, so they get the corrected paired test. ResNet only has its
published mean and 95% CI, so it gets a Welch test with both variances inflated by the same factor.

In [ ]:
records = json.load(open(RESULTS_JSON))["comprehensive"]["records"]
resnet = {(r["problem"].lower(), r["threshold"]): (r["f1"] / 100, r["err"] / 100)
          for r in records if r["approach"] == "ResNet - (Benchmark)"}

rows = []
for p, s in SETTINGS:
    tag = f"<s{s},{p.upper()}>"
    ml = fold_scores(EXP / "experiments-finetuning" / f"openai-modernbert-experiment-{p}-alpha{s}" / "logs" / "test_logs.csv")
    baselines = [("Swin", fold_scores(EXP / "experiments-swin" / f"openai-swin-experiment-{p}-alpha{s}" / "logs" / "test_logs.csv"))]
    if p == "p3":
        baselines.append(("VADER", fold_scores(EXP / "experiments-vader" / f"openai-percept_dataset_alpha{s}_p3.csv")))
    for name, scores in baselines:
        diff, p_plain, t_nb, p_nb = corrected_paired(ml, scores)
        rows.append(dict(setting=tag, baseline=name, diff=diff * 100, t_corrected=t_nb, p_plain=p_plain, p_corrected=p_nb))
    mean_r, ci_r = resnet[(p, s)]
    sd_r = ci_r * math.sqrt(K) / stats.t.ppf(0.975, K - 1)
    v_ml, v_r = ml.var(ddof=1) / K, sd_r ** 2 / K
    df_w = (v_ml + v_r) ** 2 / (v_ml ** 2 / (K - 1) + v_r ** 2 / (K - 1))
    t_plain = (ml.mean() - mean_r) / math.sqrt(v_ml + v_r)
    t_nb = t_plain / math.sqrt(INFLATION)
    rows.append(dict(setting=tag, baseline="ResNet", diff=(ml.mean() - mean_r) * 100, t_corrected=t_nb,
                     p_plain=2 * stats.t.sf(abs(t_plain), df_w), p_corrected=2 * stats.t.sf(abs(t_nb), df_w)))

ph = add_holm(pd.DataFrame(rows))
ph.round(5)

In [ ]:
print(f"max Holm-adjusted p: corrected {ph['p_holm_corrected'].max():.4f}, plain {ph['p_holm_plain'].max():.4f}")